# Pengecekan dan Ekstraksi Data SLS SE2026
Notebook ini mengekstrak data SLS (16 digit kode wilayah) dari berkas Excel progres pendataan dan pemutakhiran keluarga, membandingkannya dengan data scraped dari dashboard FASIH (`dashboard_scraped_data.csv`), dan menghasilkan berkas tabulasi final perbandingan.

Tanggal Update  : 11 Juli 2026

In [23]:
import pandas as pd
import os

# Definisikan lokasi file input
file_pendataan = r"11Juli2026/Export_Progres_Pendataan_Sub_Satuan_Lingkungan_Setempat_Sub-SLS.xlsx"
file_pemutakhiran = r"11Juli2026/Export_Progres_Pemutakhiran_Keluarga_Sub_Satuan_Lingkungan_Setempat_Sub-SLS.xlsx"
file_csv = "../../dashboard_scraped_data.csv"
file_processed_csv = "../../dashboard_scraped_data_processed.csv"
output_excel = r"8Juli2026/pengecekan_sls_perbandingan.xlsx"

# Pastikan file ada
print("File Pendataan ada:", os.path.exists(file_pendataan))
print("File Pemutakhiran ada:", os.path.exists(file_pemutakhiran))
print("File Scraped CSV ada:", os.path.exists(file_csv))
print("File Scraped Processed CSV ada:", os.path.exists(file_processed_csv))

File Pendataan ada: True
File Pemutakhiran ada: True
File Scraped CSV ada: True
File Scraped Processed CSV ada: True


## 1. Membaca dan Mengagregasikan Data Scraped (`dashboard_scraped_data.csv`)
Data di dashboard_scraped_data.csv berisi status prelist per petugas per SLS. Kita akan mengagregasikan status ini berdasarkan SLS Code untuk masing-masing Kategori petugas (Pencacah dan Pengawas) agar tidak terjadi double-counting.

In [24]:
print("Membaca data scraped...")
df_csv = pd.read_csv(file_csv)
df_csv['SLS Code'] = df_csv['SLS Code'].astype(str).str.strip()

status_cols = [
    "OPEN", "APPROVED BY Pengawas", "SUBMITTED BY Pencacah", "DRAFT",
    "REJECTED BY Pengawas", "REJECTED BY Admin Kabupaten", "REVOKED BY Pengawas",
    "SUBMITTED RESPONDENT", "COMPLETED BY Admin Kabupaten", "EDITED BY Admin Kabupaten"
]
realisasi_cols = [
    "APPROVED BY Pengawas", "SUBMITTED BY Pencacah", "REJECTED BY Pengawas", 
    "REJECTED BY Admin Kabupaten", "REVOKED BY Pengawas", "SUBMITTED RESPONDENT", 
    "COMPLETED BY Admin Kabupaten", "EDITED BY Admin Kabupaten"
]

# Pisahkan berdasarkan kategori petugas
df_csv_pcl = df_csv[df_csv['Category'] == 'Pencacah']
df_csv_pml = df_csv[df_csv['Category'] == 'Pengawas']

# Agregasikan per SLS Code
pencacah_agg = df_csv_pcl.groupby('SLS Code')[status_cols].sum()
pencacah_agg['Total_Prelist_Scraped'] = pencacah_agg[status_cols].sum(axis=1)
pencacah_agg['Realisasi_Scraped'] = pencacah_agg[realisasi_cols].sum(axis=1)
pencacah_agg = pencacah_agg.rename(columns={col: f'Scraped_Pencacah_{col}' for col in pencacah_agg.columns})

pengawas_agg = df_csv_pml.groupby('SLS Code')[status_cols].sum()
pengawas_agg['Total_Prelist_Scraped'] = pengawas_agg[status_cols].sum(axis=1)
pengawas_agg['Realisasi_Scraped'] = pengawas_agg[realisasi_cols].sum(axis=1)
pengawas_agg = pengawas_agg.rename(columns={col: f'Scraped_Pengawas_{col}' for col in pengawas_agg.columns})

print(f"Total SLS unik di Scraped (Pencacah): {len(pencacah_agg)}")
print(f"Total SLS unik di Scraped (Pengawas): {len(pengawas_agg)}")

Membaca data scraped...
Total SLS unik di Scraped (Pencacah): 736
Total SLS unik di Scraped (Pengawas): 735


## 2. Fungsi Pembantu untuk Proses Sheet Excel

In [25]:
def process_sheet(file_path, sheet_name):
    print(f"Memproses Sheet: {sheet_name}...")
    # Load excel data, baris ke-4 (indeks 3) berisi header nama kolom
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=3)
    
    # Buang baris pertama data yang berisi penomoran kolom (1), (2), dst.
    df = df.iloc[1:].reset_index(drop=True)
    
    # Rename kolom pertama menjadi 'Kode' jika belum
    df = df.rename(columns={df.columns[0]: 'Kode'})
    
    # Bersihkan Kode SLS
    df['Kode'] = df['Kode'].fillna('').astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    
    # Filter baris level SLS (Kode dengan panjang 16 digit)
    df_sls = df[df['Kode'].str.len() == 16].copy()
    
    # Gabungkan dengan data scraped
    df_merged = df_sls.merge(pencacah_agg, left_on='Kode', right_index=True, how='left')
    df_merged = df_merged.merge(pengawas_agg, left_on='Kode', right_index=True, how='left')
    
    # Isi nilai NaN dengan 0 untuk kolom-kolom scraped
    scraped_cols = list(pencacah_agg.columns) + list(pengawas_agg.columns)
    for col in scraped_cols:
        if col in df_merged.columns:
            df_merged[col] = df_merged[col].fillna(0).astype(int)
            
    # Tambah kolom selisih/diff jika sheet pendataan/pemutakhiran utama
    if sheet_name == 'PROGRES PENDATAAN':
        target_col = 'Jumlah Prelist Usaha & Keluarga'
        real_col = 'Jumlah Responden Didata'
        
        df_merged['Diff_Target_Pencacah'] = df_merged[target_col] - df_merged['Scraped_Pencacah_Total_Prelist_Scraped']
        df_merged['Diff_Realisasi_Pencacah'] = df_merged[real_col] - df_merged['Scraped_Pencacah_Realisasi_Scraped']
        
        df_merged['Diff_Target_Pengawas'] = df_merged[target_col] - df_merged['Scraped_Pengawas_Total_Prelist_Scraped']
        df_merged['Diff_Realisasi_Pengawas'] = df_merged[real_col] - df_merged['Scraped_Pengawas_Realisasi_Scraped']
        
    elif sheet_name == 'KELUARGA':
        target_col = 'Prelist Awal'
        real_col = [c for c in df_merged.columns if 'Total Hasil Pendataan' in c]
        if real_col:
            real_col = real_col[0]
            df_merged[target_col] = pd.to_numeric(df_merged[target_col], errors='coerce').fillna(0).astype(int)
            df_merged[real_col] = pd.to_numeric(df_merged[real_col], errors='coerce').fillna(0).astype(int)
            
            df_merged['Diff_Target_Pencacah'] = df_merged[target_col] - df_merged['Scraped_Pencacah_Total_Prelist_Scraped']
            df_merged['Diff_Realisasi_Pencacah'] = df_merged[real_col] - df_merged['Scraped_Pencacah_Realisasi_Scraped']
            
            df_merged['Diff_Target_Pengawas'] = df_merged[target_col] - df_merged['Scraped_Pengawas_Total_Prelist_Scraped']
            df_merged['Diff_Realisasi_Pengawas'] = df_merged[real_col] - df_merged['Scraped_Pengawas_Realisasi_Scraped']

    return df_merged

## 3. Memproses Semua Sheet dan Membandingkan Data

In [26]:
processed_sheets = {}

# 1. File Pendataan
xls_p = pd.ExcelFile(file_pendataan)
for sheet in xls_p.sheet_names:
    processed_sheets[sheet] = process_sheet(file_pendataan, sheet)

# 2. File Pemutakhiran
xls_m = pd.ExcelFile(file_pemutakhiran)
for sheet in xls_m.sheet_names:
    processed_sheets[sheet] = process_sheet(file_pemutakhiran, sheet)

Memproses Sheet: PROGRES PENDATAAN...
Memproses Sheet: USAHA PERUSAHAAN...
Memproses Sheet: SKALA USAHA...
Memproses Sheet: USAHA KELUARGA...
Memproses Sheet: JARINGAN USAHA...
Memproses Sheet: KELUARGA...
Memproses Sheet: ANGGOTA KELUARGA...


## 4. Membuat Ringkasan Tabulasi Final

In [27]:
summary_data = []

for sheet_name, df_s in processed_sheets.items():
    tot_excel_rows = len(df_s)
    
    if 'Diff_Target_Pencacah' in df_s.columns:
        diff_target_pcl_count = (df_s['Diff_Target_Pencacah'] != 0).sum()
        diff_real_pcl_count = (df_s['Diff_Realisasi_Pencacah'] != 0).sum()
        diff_target_pml_count = (df_s['Diff_Target_Pengawas'] != 0).sum()
        diff_real_pml_count = (df_s['Diff_Realisasi_Pengawas'] != 0).sum()
    else:
        diff_target_pcl_count = "N/A"
        diff_real_pcl_count = "N/A"
        diff_target_pml_count = "N/A"
        diff_real_pml_count = "N/A"
        
    summary_data.append({
        'Nama Sheet': sheet_name,
        'Total SLS (Baris)': tot_excel_rows,
        'Selisih Target (Pencacah)': diff_target_pcl_count,
        'Selisih Realisasi (Pencacah)': diff_real_pcl_count,
        'Selisih Target (Pengawas)': diff_target_pml_count,
        'Selisih Realisasi (Pengawas)': diff_real_pml_count
    })

df_summary = pd.DataFrame(summary_data)
df_summary

,Nama Sheet,Total SLS (Baris),Selisih Target (Pencacah),Selisih Realisasi (Pencacah),Selisih Target (Pengawas),Selisih Realisasi (Pengawas)
0,PROGRES PENDATAAN,737,652,131,650,129
1,USAHA PERUSAHAAN,737,N/A,N/A,N/A,N/A
2,SKALA USAHA,737,N/A,N/A,N/A,N/A
3,USAHA KELUARGA,737,N/A,N/A,N/A,N/A
4,JARINGAN USAHA,737,N/A,N/A,N/A,N/A
5,KELUARGA,736,722,476,720,475
6,ANGGOTA KELUARGA,736,N/A,N/A,N/A,N/A


## 5. Menyimpan Output Final ke Excel Multi-Sheet

In [28]:
print(f"Menyimpan hasil ke {output_excel}...")
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='SUMMARY', index=False)
    for sheet_name, df_s in processed_sheets.items():
        df_s.to_excel(writer, sheet_name=sheet_name, index=False)
print("Tabulasi perbandingan selesai disimpan.")

# Load processed CSV untuk join
print(f"\nMembaca data scraped processed dari {file_processed_csv}...")
df_processed_csv = pd.read_csv(file_processed_csv)
df_processed_csv['SLS Code'] = df_processed_csv['SLS Code'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Simpan masing-masing sheet ke file Excel terpisah (7 file) dengan join
print("\nMenyimpan masing-masing sheet ke file terpisah (7 file) dengan join...")
output_dir = "11Juli2026"
os.makedirs(output_dir, exist_ok=True)
for sheet_name, df_s in processed_sheets.items():
    # Pembersihan Kode SLS di df_s agar sinkron
    df_s_cleaned = df_s.copy()
    df_s_cleaned['Kode'] = df_s_cleaned['Kode'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    
    # Lakukan join
    df_joined = df_s_cleaned.merge(df_processed_csv, left_on='Kode', right_on='SLS Code', how='left')
    
    # Tambahkan kolom email (lowercase) sebagai salinan Email (uppercase)
    df_joined['email'] = df_joined['Email']
    
    # Susun kolom agar kolom tambahan berada di urutan akhir sesuai request
    csv_cols_to_add = [
        'email', 'Category', 'Email', 'SLS Code', 'OPEN', 'DRAFT', 
        'SUBMITTED BY Pencacah', 'REJECTED BY Pengawas', 'APPROVED BY Pengawas', 
        'nama_petugas', 'jabatan_petugas', 'nama_kec', 'koseka'
    ]
    # Saring kolom yang ada di excel (menghindari duplikasi)
    original_cols = list(df_s.columns)
    final_cols = original_cols + [col for col in csv_cols_to_add if col not in original_cols]
    df_joined = df_joined[final_cols]
    
    safe_sheet_name = "".join([c for c in sheet_name if c.isalpha() or c.isdigit() or c==' ']).strip()
    file_name = f"{safe_sheet_name}.xlsx"
    out_file = os.path.join(output_dir, file_name)
    df_joined.to_excel(out_file, index=False)
    print(f"File disimpan: {out_file} (Baris: {len(df_joined)})")

Menyimpan hasil ke 8Juli2026/pengecekan_sls_perbandingan.xlsx...
Tabulasi perbandingan selesai disimpan.

Membaca data scraped processed dari ../../dashboard_scraped_data_processed.csv...

Menyimpan masing-masing sheet ke file terpisah (7 file) dengan join...
File disimpan: 11Juli2026\PROGRES PENDATAAN.xlsx (Baris: 1472)
File disimpan: 11Juli2026\USAHA PERUSAHAAN.xlsx (Baris: 1472)
File disimpan: 11Juli2026\SKALA USAHA.xlsx (Baris: 1472)
File disimpan: 11Juli2026\USAHA KELUARGA.xlsx (Baris: 1472)
File disimpan: 11Juli2026\JARINGAN USAHA.xlsx (Baris: 1472)
File disimpan: 11Juli2026\KELUARGA.xlsx (Baris: 1471)
File disimpan: 11Juli2026\ANGGOTA KELUARGA.xlsx (Baris: 1471)


## 6. Membuat Tabulasi Agregat PCL

In [29]:
# Load processed CSV untuk info PCL & PML
print(f"Membaca data scraped processed dari {file_processed_csv}...")
df_scraped = pd.read_csv(file_processed_csv)
df_scraped['SLS Code'] = df_scraped['SLS Code'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Pisahkan PCL and PML
df_pcl_info = df_scraped[df_scraped['Category'] == 'Pencacah'].copy()
df_pml_info = df_scraped[df_scraped['Category'] == 'Pengawas'].copy()

# Rename kolom agar memperjelas peran PCL dan PML
df_pcl_info = df_pcl_info.rename(columns={
    'nama_petugas': 'Nama PCL',
    'Email': 'Email PCL',
    'OPEN': 'OPEN',
    'DRAFT': 'DRAFT',
    'SUBMITTED BY Pencacah': 'SUBMITTED BY Pencacah',
    'REJECTED BY Pengawas': 'REJECTED BY Pengawas',
    'APPROVED BY Pengawas': 'APPROVED BY Pengawas'
})

df_pml_info = df_pml_info.rename(columns={
    'nama_petugas': 'Nama PML',
    'Email': 'Email PML'
})

# Gabungkan data PCL dan PML berdasarkan SLS Code
df_scraped_merged = pd.merge(
    df_pcl_info[['SLS Code', 'Nama PCL', 'Email PCL', 'OPEN', 'DRAFT', 'SUBMITTED BY Pencacah', 'REJECTED BY Pengawas', 'APPROVED BY Pengawas', 'nama_kec', 'koseka']],
    df_pml_info[['SLS Code', 'Nama PML', 'Email PML']],
    on='SLS Code',
    how='outer'
)

# Definisikan fungsi pembantu untuk agregasi PCL
# Definisikan fungsi pembantu untuk agregasi PCL
# Definisikan fungsi pembantu untuk agregasi PCL
def aggregate_pcl(df_s, sheet_name):
    # Bersihkan Kode SLS di df_s agar sinkron
    df_s_cleaned = df_s.copy()
    df_s_cleaned['Kode'] = df_s_cleaned['Kode'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    
    # Ambil kolom numerik asli dari Excel (selain Kode dan SLS Name di kolom ke-2)
    excel_cols = list(df_s.columns)
    for col in excel_cols[2:]:
        # Bersihkan koma/titik untuk konversi ke numerik
        df_s_cleaned[col] = df_s_cleaned[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_s_cleaned[col] = pd.to_numeric(df_s_cleaned[col], errors='coerce').fillna(0)
        
    # Gabungkan dengan data scraped PCL/PML
    df_joined = df_s_cleaned.merge(df_scraped_merged, left_on='Kode', right_on='SLS Code', how='left')
    
    # Tentukan kolom numerik yang akan dijumlahkan (sum)
    numeric_cols = df_joined.select_dtypes(include=['number']).columns.tolist()
    cols_to_exclude = ['Kode', 'SLS Code']
    numeric_cols = [col for col in numeric_cols if col not in cols_to_exclude]
    
    # Isi NaN pada Email PCL dengan placeholder 'UNASSIGNED'
    df_joined['Email PCL'] = df_joined['Email PCL'].fillna('UNASSIGNED')
    
    # Helper untuk menggabungkan teks unik dengan &
    def join_unique_text(series):
        unique_vals = []
        for x in series.dropna().unique():
            val = str(x).strip()
            if val != '' and val.lower() != 'nan' and val != 'UNASSIGNED':
                unique_vals.append(val)
        if not unique_vals:
            return 'NaN'
        return ' & '.join(unique_vals)
        
    # Buat dict agregasi untuk groupby
    text_cols = ['Nama PCL', 'Nama PML', 'Email PML', 'nama_kec', 'koseka']
    agg_dict = {}
    for col in text_cols:
        if col in df_joined.columns:
            agg_dict[col] = join_unique_text
            
    for col in numeric_cols:
        agg_dict[col] = 'sum'
        
    # Group berdasarkan Email PCL
    df_agg = df_joined.groupby('Email PCL', as_index=False, dropna=False).agg(agg_dict)
    
    # Susun kolom agar kolom identitas berada di depan
    front_cols = ['Nama PCL', 'Email PCL', 'Nama PML', 'Email PML', 'nama_kec', 'koseka']
    front_cols = [col for col in front_cols if col in df_agg.columns]
    other_cols = [col for col in df_agg.columns if col not in front_cols]
    df_agg = df_agg[front_cols + other_cols]
    
    # Hitung ulang persentase agar akurat secara agregat
    if sheet_name == 'PROGRES PENDATAAN':
        if 'Jumlah Prelist Usaha & Keluarga' in df_agg.columns:
            df_agg['Persentase Responden Didata'] = (df_agg['Jumlah Responden Didata'] / df_agg['Jumlah Prelist Usaha & Keluarga'] * 100).fillna(0).round(2)
    elif sheet_name == 'USAHA PERUSAHAAN':
        if 'Total' in df_agg.columns:
            for col in ['Ditemukan', 'Tutup', 'Ganda', 'Tidak Ditemukan', 'Baru']:
                pct_col = f'Persentase {col}'
                if pct_col in df_agg.columns:
                    df_agg[pct_col] = (df_agg[col] / df_agg['Total'] * 100).fillna(0).round(2)
            if 'Persentase Total' in df_agg.columns:
                df_agg['Persentase Total'] = (df_agg['Total'] / df_agg['Total'] * 100).fillna(0).round(2)
    elif sheet_name == 'SKALA USAHA':
        if 'Jumlah Prelist UB' in df_agg.columns:
            df_agg['Persentase UB yang Berhasil Didata'] = (df_agg['Jumlah UB yang Berhasil Didata'] / df_agg['Jumlah Prelist UB'] * 100).fillna(0).round(2)
        if 'Jumlah Prelist UMKM (UM + UMK)' in df_agg.columns:
            df_agg['Persentase UMKM yang Berhasil Didata (UM + UMK)'] = (df_agg['Jumlah UMKM yang Berhasil Didata (UM + UMK)'] / df_agg['Jumlah Prelist UMKM (UM + UMK)'] * 100).fillna(0).round(2)
    elif sheet_name == 'KELUARGA':
        if 'Prelist Awal' in df_agg.columns:
            for col in ['Ditemukan', 'Meninggal', 'Tidak Eligible', 'Tidak Dapat Ditemui Sampai Akhir Pendataan', 'Tidak Ditemukan']:
                pct_col = f'Persentase {col}'
                if pct_col in df_agg.columns:
                    df_agg[pct_col] = (df_agg[col] / df_agg['Prelist Awal'] * 100).fillna(0).round(2)
            real_col = [c for c in df_agg.columns if 'Total Hasil Pendataan' in c]
            if real_col:
                real_col = real_col[0]
                pct_real_col = [c for c in df_agg.columns if 'Persentase Total Hasil Pendataan' in c]
                if pct_real_col:
                    pct_real_col = pct_real_col[0]
                    df_agg[pct_real_col] = (df_agg[real_col] / df_agg['Prelist Awal'] * 100).fillna(0).round(2)
                    
    return df_agg
# Jalankan agregasi untuk semua sheet
aggregated_sheets = {}
for sheet_name, df_s in processed_sheets.items():
    aggregated_sheets[sheet_name] = aggregate_pcl(df_s, sheet_name)

# 1. Simpan ke dalam satu file Excel multi-sheet (Tabulasi Agregat PCL)
output_agg_excel = r"11Juli2026/tabulasi_agregat_pcl.xlsx"
print(f"\nMenyimpan file agregat gabungan ke {output_agg_excel}...")
with pd.ExcelWriter(output_agg_excel, engine='openpyxl') as writer:
    for sheet_name, df_agg in aggregated_sheets.items():
        df_agg.to_excel(writer, sheet_name=sheet_name, index=False)
print("File agregat gabungan selesai disimpan.")

# 2. Simpan masing-masing agregat sheet ke file Excel terpisah (7 file)
print("\nMenyimpan masing-masing agregat sheet ke file terpisah (7 file)...")
for sheet_name, df_agg in aggregated_sheets.items():
    safe_sheet_name = "".join([c for c in sheet_name if c.isalpha() or c.isdigit() or c==' ']).strip()
    file_name = f"Agregat_PCL_{safe_sheet_name}.xlsx"
    out_file = os.path.join(output_dir, file_name)
    df_agg.to_excel(out_file, index=False)
    print(f"File disimpan: {out_file} (Baris: {len(df_agg)})")

# 3. Pengecekan email PCL yang double pada hasil agregat
print("\n=== Mengecek Email PCL yang Double pada Hasil Agregat ===")
for sheet_name, df_agg in aggregated_sheets.items():
    df_temp = df_agg.dropna(subset=['Email PCL']).copy()
    email_counts = df_temp['Email PCL'].value_counts()
    double_emails = email_counts[email_counts > 1].index.tolist()
    
    if double_emails:
        print(f"\nSheet '{sheet_name}': Ditemukan {len(double_emails)} email PCL yang double/muncul lebih dari sekali:")
        df_double = df_temp[df_temp['Email PCL'].isin(double_emails)].sort_values('Email PCL')
        cols_to_show = ['Email PCL', 'Nama PCL', 'Nama PML', 'Email PML', 'nama_kec', 'koseka']
        cols_to_show = [c for c in cols_to_show if c in df_double.columns]
        print(df_double[cols_to_show].to_string(index=False))
    else:
        print(f"\nSheet '{sheet_name}': Tidak ada email PCL yang double.")


Membaca data scraped processed dari ../../dashboard_scraped_data_processed.csv...

Menyimpan file agregat gabungan ke 11Juli2026/tabulasi_agregat_pcl.xlsx...
File agregat gabungan selesai disimpan.

Menyimpan masing-masing agregat sheet ke file terpisah (7 file)...
File disimpan: 11Juli2026\Agregat_PCL_PROGRES PENDATAAN.xlsx (Baris: 131)
File disimpan: 11Juli2026\Agregat_PCL_USAHA PERUSAHAAN.xlsx (Baris: 131)
File disimpan: 11Juli2026\Agregat_PCL_SKALA USAHA.xlsx (Baris: 131)
File disimpan: 11Juli2026\Agregat_PCL_USAHA KELUARGA.xlsx (Baris: 131)
File disimpan: 11Juli2026\Agregat_PCL_JARINGAN USAHA.xlsx (Baris: 131)
File disimpan: 11Juli2026\Agregat_PCL_KELUARGA.xlsx (Baris: 130)
File disimpan: 11Juli2026\Agregat_PCL_ANGGOTA KELUARGA.xlsx (Baris: 130)

=== Mengecek Email PCL yang Double pada Hasil Agregat ===

Sheet 'PROGRES PENDATAAN': Tidak ada email PCL yang double.

Sheet 'USAHA PERUSAHAAN': Tidak ada email PCL yang double.

Sheet 'SKALA USAHA': Tidak ada email PCL yang double.

She